# Data Cleaning — DaimonUpDown Workforce Project

This notebook documents the data-cleaning work performed during the MariaDB loading and validation process.

## Workflow

Raw CSV → MariaDB load → Data-quality checks → Identify problems → Clean/reload → Validate again


## 1. Problems found during loading

### Onboarding
- Empty `onboarding_end_date` values caused date-truncation warnings.
- Empty `onboarding_satisfaction` values caused incorrect-integer warnings.
- The load was corrected so empty values become SQL `NULL` where appropriate.

### Offboarding
- Boolean values arrived from CSV as `True` and `False`.
- Empty boolean/numeric values also caused warnings.
- The load was corrected to convert these values before insertion.

### General date handling
- Empty CSV date values must not be inserted as empty strings into typed date columns.
- `NULLIF(value, '')` was used where appropriate to convert empty CSV values to SQL `NULL`.


## 2. Onboarding cleaning

The corrected loading process handles nullable fields through variables and conversion logic.

Example pattern:

```sql
SET onboarding_end_date = NULLIF(@onboarding_end_date, '');
```

Empty satisfaction values are also converted appropriately instead of forcing `''` into an integer column.

### Validation

Final onboarding row count:

```text
1,500
```


## 3. Offboarding cleaning

The source contains boolean values such as:

```text
True
False
```

These cannot be loaded directly into the integer/boolean database fields in the original form.

The corrected load converts them to the database representation and converts empty values to `NULL` where allowed.

### Validation

Final offboarding row count:

```text
320
```


## 4. Data-quality checks after cleaning

After the corrected reload, the database was checked for:

- Duplicate primary IDs
- Orphaned foreign-key records
- NULLs in important key fields
- Invalid date relationships
- Valid category/status values

### Results

All duplicate-ID checks returned `0`.

All checked orphan-record queries returned `0`.

The checked key fields contained `0` unexpected NULLs.


## 5. Final validated database counts

| Table | Rows |
|---|---:|
| applications | 2,600 |
| assignments | 2,341 |
| attendance | 22,567 |
| candidates | 2,000 |
| client_feedback | 5,895 |
| clients | 300 |
| compensation_history | 10 |
| departments | 9 |
| employees | 1,600 |
| employee_surveys | 11,956 |
| employment_history | 1,600 |
| locations | 486 |
| offboarding | 320 |
| onboarding | 1,500 |
| performance | 3,029 |
| positions | 6 |
| qualifications | 1,904 |
| schools | 486 |
| training | 9,738 |
| visa_history | 1,500 |


## 6. Cleaning decisions

| Problem | Action | Reason |
|---|---|---|
| Empty date | Convert empty string to SQL `NULL` | Preserve nullable date semantics |
| Empty integer | Convert to SQL `NULL` where nullable | Avoid invalid integer values |
| `True` / `False` strings | Normalize to database boolean/integer values | Match schema |
| Duplicate IDs | Investigate and validate | Protect primary-key integrity |
| Orphan records | Check foreign-key relationships | Protect relational integrity |

The important principle is: **clean the data without inventing values**. Missing source information remains missing (`NULL`) rather than being guessed.


## 7. Reproducibility

The database loading and correction logic is stored in the project's SQL directory:

```text
sql/03_load_data.sql
sql/04_data_quality_checks.sql
sql/06_onboarding_offboarding.sql
```

This notebook documents what was cleaned and why. The SQL files contain the reproducible database operations.
